
# Phase 1 — USA Fortnight Feature Extraction

Extract 15-day (fortnight) satellite features per sugarcane field for **Texas, Louisiana, and Florida**.

**Output:** `usa_fortnight_features.csv` (downloads in Colab)

Run in Google Colab or any environment with Earth Engine access.


In [2]:

# --- CONFIG ---
EE_PROJECT = 'plenary-matrix-441317-c4'
EE_ASSET = 'projects/ee-junaidali101452/assets/FL_TX_LA_Geometries_Confidence'

# Pilot: set MAX_FIELDS=10 for a quick test; None for full 33,241-field extraction
MAX_FIELDS = None

# Optional: 'Texas', 'Louisiana', 'Florida', or None for all states
STATE_FILTER = 'Texas'

# Fortnight extraction years (aligned with existing FYP Texas pipeline)
COLLECTION_YEARS = [
    {'start_date': '2019-01-01', 'end_date': '2019-12-31', 'year': 2019},
    {'start_date': '2021-01-01', 'end_date': '2021-12-31', 'year': 2021},
    {'start_date': '2022-01-01', 'end_date': '2022-12-31', 'year': 2022},
]

NUM_SPLITS = 10
OUTPUT_CSV = 'usa_fortnight_features.csv'

# Asset property `label` → state name (Florida sample field has label=2)
STATE_LABEL_MAP = {0: 'Texas', 1: 'Louisiana', 2: 'Florida'}

In [3]:

import ee
import pandas as pd
from datetime import datetime, timedelta

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

if IN_COLAB:
    drive.mount('/content/drive')

fc = ee.FeatureCollection(EE_ASSET)
print('Total fields in asset:', fc.size().getInfo())


Mounted at /content/drive
Total fields in asset: 33241


In [4]:

# State annual sugarcane yields (tons/acre) — USDA NASS reference for Phase 3 mapping
STATE_ANNUAL_YIELDS = {
    'Florida':   {2017: 41.1, 2018: 41.9, 2019: 43.0, 2021: 42.6, 2022: 44.6},
    'Louisiana': {2017: 32.8, 2018: 35.4, 2019: 28.1, 2021: 29.3, 2022: 32.3},
    'Texas':     {2017: 37.1, 2018: 36.6, 2019: 33.8, 2021: 30.9, 2022: 22.6},
}

state_yield_lookup = {
    (state, year): yield_tpa
    for state, years in STATE_ANNUAL_YIELDS.items()
    for year, yield_tpa in years.items()
}

pd.DataFrame(
    [(s, y, v) for s, years in STATE_ANNUAL_YIELDS.items() for y, v in years.items()],
    columns=['state', 'year', 'state_annual_yield'],
)

# Preview one field and verify label → state mapping
sample = fc.first().getInfo()
print('Sample properties:', sample.get('properties'))
print('Sample coords:', sample['geometry']['coordinates'][0][0])


Sample properties: {'area': 0.22279233590870987, 'label': 2, 'mean': 45}
Sample coords: [-81.45762288930143, 26.63840662272212]


In [5]:

def generate_fortnight_starts(start_date, end_date):
    """Return fortnight start dates (1st and 15th of each month) within range."""
    dates = []
    current = datetime.strptime(start_date, '%Y-%m-%d')
    end = datetime.strptime(end_date, '%Y-%m-%d')
    while current <= end:
        dates.append(current.strftime('%Y-%m-%d'))
        if current.day == 1:
            current = current.replace(day=15)
        elif current.month == 12:
            break
        else:
            current = (current.replace(day=28) + timedelta(days=4)).replace(day=1)
    return dates

fortnight_periods = []
for entry in COLLECTION_YEARS:
    for start in generate_fortnight_starts(entry['start_date'], entry['end_date']):
        fortnight_periods.append({
            'fortnight_start': start,
            'year': entry['year'],
        })

print(f'Total fortnight windows: {len(fortnight_periods)}')
fortnight_periods[:5], '...', fortnight_periods[-2:]


Total fortnight windows: 72


([{'fortnight_start': '2019-01-01', 'year': 2019},
  {'fortnight_start': '2019-01-15', 'year': 2019},
  {'fortnight_start': '2019-02-01', 'year': 2019},
  {'fortnight_start': '2019-02-15', 'year': 2019},
  {'fortnight_start': '2019-03-01', 'year': 2019}],
 '...',
 [{'fortnight_start': '2022-12-01', 'year': 2022},
  {'fortnight_start': '2022-12-15', 'year': 2022}])

In [6]:

# Split FeatureCollection for manageable getInfo() batches
total_count = fc.size().getInfo()
split_count = int(total_count / NUM_SPLITS)
field_splits = []

for i in range(NUM_SPLITS):
    start_index = i * split_count
    if i == NUM_SPLITS - 1:
        current_count = total_count - start_index
    else:
        current_count = split_count
    field_splits.append(fc.toList(current_count, start_index))

print(f'Split {total_count} fields into {NUM_SPLITS} batches (~{split_count} each)')


Split 33241 fields into 10 batches (~3324 each)


In [7]:

# --- Earth Engine collections (static references) ---
NASA_srtm = ee.Image('USGS/SRTMGL1_003')


def mask_clouds(image):
    qa60 = image.select('QA60')
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa60.bitwiseAnd(cloud_bit_mask).eq(0).And(qa60.bitwiseAnd(cirrus_bit_mask).eq(0))
    return image.updateMask(mask).copyProperties(image, ['system:time_start'])


def calculate_indices(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    gndvi = image.normalizedDifference(['B8', 'B3']).rename('GNDVI')
    evi = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {'NIR': image.select('B8'), 'RED': image.select('B4'), 'BLUE': image.select('B2')}
    ).rename('EVI')
    red = image.select('B4')
    nir = image.select('B8')
    savi = nir.subtract(red).divide(nir.add(red).add(0.5)).multiply(1.5).rename('SAVI')
    vci = ndvi.expression('(NDVI - 0) / (0.8 - 0) * 100', {'NDVI': ndvi}).rename('VCI')
    tvi = image.expression('0.5 * (NIR - RED) / (NIR + RED + 0.5)', {'NIR': nir, 'RED': red}).rename('TVI')
    bi = image.expression('sqrt((RED*RED) + (NIR*NIR))', {'RED': red, 'NIR': nir}).rename('BI')
    bi2 = image.expression('(RED + NIR) / 2', {'RED': red, 'NIR': nir}).rename('BI2')
    ci = image.expression('(NIR / RED) - 1', {'NIR': nir, 'RED': red}).rename('CI')
    ci1 = image.expression('(RED / NIR) - 1', {'RED': red, 'NIR': nir}).rename('CI1')
    satvi = image.expression('(NIR - RED - 0.5) / (NIR + RED + 0.5)', {'NIR': nir, 'RED': red}).rename('SATVI')
    hvsi = image.expression(
        'sqrt((NIR - RED)*(NIR - RED) + (NIR - GREEN)*(RED - GREEN))',
        {'NIR': nir, 'RED': red, 'GREEN': image.select('B3')}
    ).rename('HVSI')
    soci = image.expression('(NIR - RED) / (NIR + RED)', {'NIR': nir, 'RED': red}).rename('SOCI')
    asi = image.expression('(NIR - RED) / (NIR + RED + 0.5)', {'NIR': nir, 'RED': red}).rename('ASI')
    bsi = image.expression('((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))', {
        'SWIR': image.select('B11'), 'RED': red, 'NIR': nir, 'BLUE': image.select('B2')
    }).rename('BSI')
    msavi = image.expression(
        '(2 * NIR + 1 - sqrt((2 * NIR + 1)*(2 * NIR + 1) - 8 * (NIR - RED))) / 2',
        {'NIR': nir, 'RED': red}
    ).rename('MSAVI')
    return image.addBands([ndvi, gndvi, evi, savi, vci, tvi, bi, bi2, ci, ci1, satvi, hvsi, soci, asi, bsi, msavi])


def cloud_mask_landsat(image):
    qa = image.select('QA_PIXEL')
    cloud = 1 << 3
    cirrus = 1 << 9
    mask = qa.bitwiseAnd(cloud).eq(0).And(qa.bitwiseAnd(cirrus).eq(0))
    return image.updateMask(mask)


def calculate_indices_landsat(img):
    nbr = img.normalizedDifference(['SR_B5', 'SR_B7']).rename('NBR')
    ndmi = img.normalizedDifference(['SR_B5', 'SR_B6']).rename('NDMI')
    ndwi = img.normalizedDifference(['SR_B3', 'SR_B5']).rename('NDWI')
    ndbi = img.normalizedDifference(['SR_B6', 'SR_B5']).rename('NDBI')
    ndbai = img.normalizedDifference(['SR_B6', 'SR_B7']).rename('NDBaI')
    mndwi = img.normalizedDifference(['SR_B3', 'SR_B6']).rename('MNDWI')
    return img.addBands([nbr, ndmi, ndwi, ndbi, ndbai, mndwi])


S2_BANDS = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'QA60']
S2_INDEX_BANDS = ['NDVI', 'EVI', 'GNDVI', 'SAVI', 'VCI', 'TVI', 'BI', 'BI2', 'CI', 'CI1', 'SATVI', 'HVSI', 'SOCI', 'ASI', 'BSI', 'MSAVI']
LS_BANDS = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'NBR', 'NDMI', 'NDWI', 'NDBI', 'NDBaI', 'MNDWI']


In [8]:

def reduce_or_none(image, roi, scale=30):
    if image is None:
        return {}
    try:
        stats = image.clip(roi).reduceRegion(reducer=ee.Reducer.mean(), geometry=roi, scale=scale)
        return stats.getInfo() or {}
    except ee.EEException as exc:
        print('reduceRegion failed:', exc)
        return {}


def build_period_imagery(start_date_str, end_date_str):
    sentinel2_period = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterDate(start_date_str, end_date_str)
    landsat8_period = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate(start_date_str, end_date_str)
    landsat9_period = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').filterDate(start_date_str, end_date_str)
    modis_period = ee.ImageCollection('MODIS/061/MCD15A3H').filterDate(start_date_str, end_date_str)

    landsat8_processed = landsat8_period.map(cloud_mask_landsat).map(calculate_indices_landsat)
    landsat9_processed = landsat9_period.map(cloud_mask_landsat).map(calculate_indices_landsat)
    sentinel2_processed = sentinel2_period.select(S2_BANDS).map(mask_clouds).map(calculate_indices)

    landsat8_imagery = None
    if landsat8_processed.size().getInfo() > 0:
        landsat8_imagery = landsat8_processed.median().select(LS_BANDS)

    landsat9_imagery = None
    if landsat9_processed.size().getInfo() > 0:
        landsat9_imagery = landsat9_processed.median().select(LS_BANDS)

    modis_imagery = None
    if modis_period.size().getInfo() > 0:
        modis_imagery = modis_period.median().select(['Lai'])

    usda_imagery = None
    usda_period = ee.ImageCollection('NASA_USDA/HSL/SMAP10KM_soil_moisture').filterDate(start_date_str, end_date_str)
    if usda_period.size().getInfo() > 0:
        usda_imagery = usda_period.median().select(['ssm'])

    return {
        'sentinel2_processed': sentinel2_processed,
        'landsat8_imagery': landsat8_imagery,
        'landsat9_imagery': landsat9_imagery,
        'modis_imagery': modis_imagery,
        'usda_imagery': usda_imagery,
    }


def extract_field_period(field_id, state, area_acres, roi, period, imagery):
    start_date_str = period['fortnight_start']
    start_dt = datetime.strptime(start_date_str, '%Y-%m-%d')
    end_date_str = (start_dt + timedelta(days=14)).strftime('%Y-%m-%d')
    year = period['year']
    state_annual_yield = state_yield_lookup.get((state, year))

    row = {
        'field_id': field_id,
        'state': state,
        'year': year,
        'fortnight_start': start_date_str,
        'fortnight_end': end_date_str,
        'state_annual_yield': state_annual_yield,
        'Area': area_acres,
    }

    usda_stats = reduce_or_none(imagery['usda_imagery'], roi)
    row['MOISTURE'] = usda_stats.get('ssm')

    sentinel2_processed = imagery['sentinel2_processed']
    sentinel_roi = sentinel2_processed.filterBounds(roi)
    s2_count = sentinel_roi.size().getInfo()

    if s2_count > 0:
        elevation = NASA_srtm.clip(roi).log().divide(10).clamp(0, 1).toFloat().rename('elevation')
        sentinel_image = sentinel_roi.median().select(S2_BANDS + S2_INDEX_BANDS).addBands(elevation)
        s2_stats = reduce_or_none(sentinel_image, roi)
        for band in S2_BANDS + S2_INDEX_BANDS:
            row[band] = s2_stats.get(band)
        row['Elevation'] = s2_stats.get('elevation')
        row['s2_scene_count'] = s2_count
    else:
        for band in S2_BANDS + S2_INDEX_BANDS:
            row[band] = None
        elevation = NASA_srtm.clip(roi).log().divide(10).clamp(0, 1).toFloat().rename('elevation')
        row['Elevation'] = reduce_or_none(elevation, roi).get('elevation')
        row['s2_scene_count'] = 0
        print(f'Warning: no Sentinel-2 for field {field_id} ({state}) period {start_date_str}')

    l8 = reduce_or_none(imagery['landsat8_imagery'], roi)
    l9 = reduce_or_none(imagery['landsat9_imagery'], roi)
    modis = reduce_or_none(imagery['modis_imagery'], roi)

    row['SR8_B1'] = l8.get('SR_B1'); row['SR8_B2'] = l8.get('SR_B2'); row['SR8_B3'] = l8.get('SR_B3')
    row['SR8_B4'] = l8.get('SR_B4'); row['SR8_B5'] = l8.get('SR_B5'); row['SR8_B6'] = l8.get('SR_B6'); row['SR8_B7'] = l8.get('SR_B7')
    row['NBR_8'] = l8.get('NBR'); row['NDMI_8'] = l8.get('NDMI'); row['NDWI_8'] = l8.get('NDWI')
    row['NDBI_8'] = l8.get('NDBI'); row['NDBaI_8'] = l8.get('NDBaI'); row['MNDWI_8'] = l8.get('MNDWI')

    row['SR9_B1'] = l9.get('SR_B1'); row['SR9_B2'] = l9.get('SR_B2'); row['SR9_B3'] = l9.get('SR_B3')
    row['SR9_B4'] = l9.get('SR_B4'); row['SR9_B5'] = l9.get('SR_B5'); row['SR9_B6'] = l9.get('SR_B6'); row['SR9_B7'] = l9.get('SR_B7')
    row['NBR_9'] = l9.get('NBR'); row['NDMI_9'] = l9.get('NDMI'); row['NDWI_9'] = l9.get('NDWI')
    row['NDBI_9'] = l9.get('NDBI'); row['NDBaI_9'] = l9.get('NDBaI'); row['MNDWI_9'] = l9.get('MNDWI')

    row['LAI'] = modis.get('Lai')
    return row


In [ ]:

# Build field list first (stable field_id per polygon)
fields_to_process = []
field_id = 0

for field_split in field_splits:
    for record in field_split.getInfo():
        props = record.get('properties', {})
        state = STATE_LABEL_MAP.get(props.get('label'), 'Unknown')
        if STATE_FILTER and state != STATE_FILTER:
            continue

        field_id += 1
        if MAX_FIELDS is not None and field_id > MAX_FIELDS:
            break
        fields_to_process.append((field_id, record, state, props))

    if MAX_FIELDS is not None and field_id >= MAX_FIELDS:
        break

print(f'Fields queued: {len(fields_to_process)}')

feature_data = []

for period in fortnight_periods:
    start_date_str = period['fortnight_start']
    end_date_str = (datetime.strptime(start_date_str, '%Y-%m-%d') + timedelta(days=14)).strftime('%Y-%m-%d')
    print(f'Period {start_date_str} → {end_date_str}')
    imagery = build_period_imagery(start_date_str, end_date_str)

    for fid, record, state, props in fields_to_process:
        polygon = ee.Feature(record)
        roi = polygon.geometry()
        area_acres = props.get('area')
        if area_acres is None:
            area_acres = roi.area().divide(4046.86).getInfo()

        row = extract_field_period(fid, state, area_acres, roi, period, imagery)
        feature_data.append(row)

print(f'Done: {len(fields_to_process)} fields, {len(feature_data)} total rows')


Fields queued: 0
Period 2019-01-01 → 2019-01-15


/usr/local/lib/python3.12/dist-packages/ee/deprecation.py:215: DeprecationWarning: 

Attention required for NASA_USDA/HSL/SMAP10KM_soil_moisture! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by NASA/SMAP/SPL4SMGP/008

Learn more: https://developers.google.com/earth-engine/datasets/catalog/NASA_USDA_HSL_SMAP10KM_soil_moisture

  warnings.warn(warning, category=DeprecationWarning)


Period 2019-01-15 → 2019-01-29
Period 2019-02-01 → 2019-02-15
Period 2019-02-15 → 2019-03-01
Period 2019-03-01 → 2019-03-15
Period 2019-03-15 → 2019-03-29
Period 2019-04-01 → 2019-04-15
Period 2019-04-15 → 2019-04-29
Period 2019-05-01 → 2019-05-15
Period 2019-05-15 → 2019-05-29
Period 2019-06-01 → 2019-06-15
Period 2019-06-15 → 2019-06-29
Period 2019-07-01 → 2019-07-15
Period 2019-07-15 → 2019-07-29
Period 2019-08-01 → 2019-08-15
Period 2019-08-15 → 2019-08-29
Period 2019-09-01 → 2019-09-15
Period 2019-09-15 → 2019-09-29
Period 2019-10-01 → 2019-10-15
Period 2019-10-15 → 2019-10-29
Period 2019-11-01 → 2019-11-15
Period 2019-11-15 → 2019-11-29
Period 2019-12-01 → 2019-12-15
Period 2019-12-15 → 2019-12-29
Period 2021-01-01 → 2021-01-15
Period 2021-01-15 → 2021-01-29
Period 2021-02-01 → 2021-02-15
Period 2021-02-15 → 2021-03-01
Period 2021-03-01 → 2021-03-15
Period 2021-03-15 → 2021-03-29
Period 2021-04-01 → 2021-04-15
Period 2021-04-15 → 2021-04-29
Period 2021-05-01 → 2021-05-15
Period 2

In [ ]:

df = pd.DataFrame(feature_data)
print(df.shape)
df.head()


In [ ]:

df.to_csv(OUTPUT_CSV, index=False)
print('Saved:', OUTPUT_CSV)

if IN_COLAB:
    from google.colab import files
    files.download(OUTPUT_CSV)



## Feature weights (Phase 3 reference)

Used later in yield mapping — not applied during extraction.


In [ ]:

feature_weights = {
    'field_id': 0,
    'state': 0,
    'fortnight_start': 0,
    'fortnight_end': 0,
    'NDVI': 0.15,
    'EVI': 0.10,
    'GNDVI': 0.05,
    'SAVI': 0.05,
    'B1': 0.03, 'B2': 0.03, 'B3': 0.03, 'B4': 0.03,
    'B5': 0.03, 'B6': 0.03, 'B7': 0.03, 'B8': 0.03,
    'B9': 0.03, 'B11': 0.03, 'B12': 0.03,
    'SR8_B1': 0.03, 'SR8_B2': 0.03, 'SR8_B3': 0.03, 'SR8_B4': 0.03,
    'SR8_B5': 0.03, 'SR8_B6': 0.03, 'SR8_B7': 0.03,
    'NBR_8': 0.02, 'NDMI_8': 0.02, 'NDWI_8': 0.02,
    'NDBI_8': 0.02, 'NDBaI_8': 0.02, 'MNDWI_8': 0.02,
    'SR9_B1': 0.02, 'SR9_B2': 0.02, 'SR9_B3': 0.02, 'SR9_B4': 0.02,
    'SR9_B5': 0.02, 'SR9_B6': 0.02, 'SR9_B7': 0.02,
    'NBR_9': 0.02, 'NDMI_9': 0.02, 'NDWI_9': 0.02,
    'NDBI_9': 0.02, 'NDBaI_9': 0.02, 'MNDWI_9': 0.02,
    'LAI': 0.05,
    'MOISTURE': 0.05,
    'Area': 0.20,
    'Elevation': 0.10,
    'VCI': 0.10, 'TVI': 0.10,
    'BI': 0.05, 'BI2': 0.05,
    'CI': 0.05, 'CI1': 0.05,
    'SATVI': 0.10, 'HVSI': 0.10,
    'SOCI': 0.05, 'ASI': 0.05,
    'BSI': 0.05, 'MSAVI': 0.10,
}
sum(feature_weights.values())
